# 03 データをまとめて地図にしよう:主成分分析(PCA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cogproject/stats-ml-bio-study/blob/main/notebooks/03_pca.ipynb)

**今日のゴール(30分)**
1. 500個の遺伝子の数字を、たった2つの数字にまとめて、801人を1枚の「地図」に置く
2. 地図の上で、がんの種類ごとに分かれるかを見る
3. ほかの人から大きくはなれた人(外れたサンプル)を、**先に決めたルール** でさがす

データは 01・02 と同じ、がんの患者さん801人の遺伝子データです。

## 0. 準備(01・02 と同じ。まとめて実行、1〜2分)

In [ ]:
!pip -q install japanize-matplotlib
import japanize_matplotlib
import urllib.request, zipfile, tarfile, glob, os
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns

os.makedirs("data", exist_ok=True)
url = "https://archive.ics.uci.edu/static/public/401/gene+expression+cancer+rna+seq.zip"
zip_path = "data/rnaseq.zip"
if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall("data")
for tgz in glob.glob("data/**/*.tar.gz", recursive=True):
    with tarfile.open(tgz) as t:
        t.extractall("data")
X = pd.read_csv(glob.glob("data/**/data.csv", recursive=True)[0], index_col=0)
y = pd.read_csv(glob.glob("data/**/labels.csv", recursive=True)[0], index_col=0)["Class"]
X_top = X[X.var().sort_values(ascending=False).index[:500]]
print("準備できました:", X_top.shape)

## 1. 主成分分析ってなに?
1人の患者さんには、500個の数字(遺伝子の働きぐあい)があります。
グラフは縦と横の2つまでしか描けないので、このままでは801人を1枚の図にできません。

**主成分分析** は、たくさんの数字を、**大事な情報をなるべく残したまま**、少ない数の「まとめの数字」に置きかえる方法です。

> **たとえ:影絵**
> 立体の物に光を当てると、壁に平らな影ができます。当てる向きを工夫すると、影を見ただけで「これはウサギだ」と分かる向きがあります。
> 主成分分析は、500次元(500個の数字)のデータに、**いちばん形が分かりやすい向きから光を当てて**、2次元の影をつくる方法です。

- いちばん「ばらつき(人によるちがい)」が大きく見える向き → **第1主成分(PC1)**
- それと直角で、次にばらつきが大きい向き → **第2主成分(PC2)**

大事な点:主成分分析は **がんの種類(答え)を見ずに** 計算します。答えは、あとで色をぬるときにだけ使います。

## 2. 下ごしらえ:標準化
遺伝子によって、数字の大きさ(0〜15くらいの物もあれば、0〜2くらいの物もある)がちがいます。
そのままだと、数字の大きい遺伝子ばかりが重視されてしまいます。

そこで、どの遺伝子も「平均 0、ばらつき 1」にそろえます。これを **標準化** といいます。
(テストの点を偏差値にして、教科ごとの難しさのちがいをそろえるのと似ています)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

Z = StandardScaler().fit_transform(X_top)   # 標準化
pca = PCA(n_components=10)                  # まとめの数字を10個つくる
P = pca.fit_transform(Z)                    # 801人 × 10個
pcs = pd.DataFrame(P[:, :2], index=X_top.index, columns=["PC1", "PC2"])
print(pcs.head())

## 3. 地図をかく:がんの種類で色分け

In [ ]:
sns.scatterplot(data=pcs, x="PC1", y="PC2", hue=y, s=12)
plt.title("主成分分析の地図(1つの点が1人)");

**読み方**:近くにいる点どうしは、遺伝子の働き方が似ている人です。

🤔 **考えてみよう**
- がんの種類ごとに、かたまりに分かれていますか?
- 02 で「KIRC だけ特別」だったのを思い出してください。KIRC はこの地図のどこにいますか?
- 重なっているのは、どのがんとどのがんですか?

## 4. 2つの数字で、どれくらい情報が残った?(寄与率)
**寄与率(きよりつ)** は、それぞれの主成分が、もとのばらつき全体の何%を表しているか、という数字です。

In [ ]:
ratio = pca.explained_variance_ratio_
for i, r in enumerate(ratio, 1):
    print(f"PC{i}: {r*100:.1f}%   (ここまでの合計 {ratio[:i].sum()*100:.1f}%)")
plt.bar(range(1, 11), ratio * 100)
plt.xlabel("主成分の番号"); plt.ylabel("寄与率(%)"); plt.title("各主成分の寄与率");

**読み方**:PC1 と PC2 の合計が、地図に残せた情報の割合です。
100% でなくても、がんの種類が分かれて見えるなら、「大事なちがい」はちゃんと残っている、ということです。

## 5. PC1 は、どの遺伝子でできている?(負荷量)
主成分は、500個の遺伝子に「重み」をかけて足し合わせたものです。この重みを **負荷量(ふかりょう)** といいます。
重みが大きい遺伝子ほど、その主成分に強く効いています。

In [ ]:
load = pd.Series(pca.components_[0], index=X_top.columns)
top = load.abs().sort_values(ascending=False).index[:10]
print("PC1 に強く効いている遺伝子 トップ10:")
print(load[top].round(3))
for g in ["gene_219", "gene_220"]:
    if g in load.index:
        print(f"{g} の PC1 での順位:{load.abs().rank(ascending=False)[g]:.0f} / 500")

🤔 02 で見つけた gene_219・gene_220 は、PC1 で上位にいますか?
上位にいれば、PC1 は「腎臓のがんかどうか」に関係する向きかもしれません(3 の地図で、KIRC が PC1 のどちらのはしにいるかと合わせて考えてみましょう)。

## 6. 外れたサンプルをさがす(ルールは先に決める)
02 で、「外れ値を外すかどうかは、データを見る前にルールを決める」と学びました。そこで、**先に** ルールを書いておきます。

> **ルール**:同じがんの種類の人たちの真ん中(重心)から、地図(PC1〜PC10)の上での距離をはかる。
> その種類の中で、距離が「平均 + 3 × 標準偏差」をこえた人を、**外れたサンプル候補** とする。
> 候補は **外さずに印をつけるだけ** にして、理由を考える。

In [ ]:
P10 = pd.DataFrame(P, index=X_top.index)
dist = pd.Series(index=X_top.index, dtype=float)
for k, idx in y.groupby(y).groups.items():
    center = P10.loc[idx].mean()
    dist[idx] = np.sqrt(((P10.loc[idx] - center) ** 2).sum(axis=1))
limit = dist.groupby(y).transform(lambda d: d.mean() + 3 * d.std())
flag = dist > limit
print("外れたサンプル候補の数:", flag.sum())
print(pd.DataFrame({"がんの種類": y[flag], "距離": dist[flag].round(1), "基準": limit[flag].round(1)}))

In [ ]:
sns.scatterplot(data=pcs, x="PC1", y="PC2", hue=y, s=12)
plt.scatter(pcs.loc[flag, "PC1"], pcs.loc[flag, "PC2"], s=80,
            facecolors="none", edgecolors="black", label="外れ候補")
plt.legend(); plt.title("外れたサンプル候補(黒い丸)");

🤔 **考えてみよう**
- 候補の人は、地図のどこにいますか? ほかのがんのかたまりの近くにいませんか?
- もしそうなら、考えられる理由は:
  - ラベル(がんの種類)の付けまちがい
  - がんが別の場所から広がってきたもの(転移)
  - 測定の失敗
  - 本当にめずらしいタイプのがん
- データだけでは、どれが正しいかは決められません。研究では、元の記録や、ほかの検査結果と照らし合わせて確かめます。

⚠️ 黒い丸は2次元の地図に描いていますが、距離は PC1〜PC10 で計算しています。だから、地図の上では真ん中にいるように見える人が候補になることもあります。

## 今日のまとめ
- 主成分分析は、たくさんの数字を、大事なちがいを残して少ない数字にまとめる方法(影絵のたとえ)
- 計算の前に標準化して、遺伝子ごとの数字の大きさをそろえる
- 寄与率で「どれだけ情報が残ったか」、負荷量で「どの遺伝子が効いているか」が分かる
- がんの種類を見ずに計算したのに、地図の上でがんの種類が分かれた → 遺伝子の働き方は、がんの種類で大きくちがう
- 外れたサンプルは、先に決めたルールで探し、外さずに理由を考える

**次回(04)**:答え(がんの種類)を使わずに、データだけで「なかま分け」をする **クラスター分析** をやります。

### もっと知りたい人へ(公式の説明)
- scikit-learn の主成分分析:https://scikit-learn.org/stable/modules/decomposition.html#pca
- scikit-learn の標準化 `StandardScaler`:https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html